# 01 — EDA pipeline (ipykeep example)

Domain-neutral exploratory pipeline. Stages are labeled; cells marked **EXPENSIVE** simulate slow I/O or compute via `time.sleep`.

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import sqlite3

# --- setup: generate the source data file (cheap) ---
DATA_DIR = Path("examples/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "sample.csv"
rng = np.random.default_rng(0)
_n = 200
_raw = pd.DataFrame({
    "id": np.arange(_n),
    "group": rng.choice(["a", "b", "c"], size=_n),
    "value": rng.normal(10, 3, size=_n).round(2),
    "score": rng.integers(0, 100, size=_n),
})
_raw.loc[rng.choice(_n, 8, replace=False), "value"] = np.nan
_raw.to_csv(DATA_PATH, index=False)
print("wrote", DATA_PATH, _raw.shape)

## Stage 1 — Load  (EXPENSIVE: simulates slow I/O)

In [ ]:
# EXPENSIVE: reads from disk; simulates slow I/O
time.sleep(0.3)
df = pd.read_csv("examples/data/sample.csv")
print("loaded", df.shape)

## Stage 2 — Clean  (EXPENSIVE: simulates compute)

In [ ]:
# EXPENSIVE: cleaning + feature engineering
time.sleep(0.3)
df_clean = (
    df.dropna(subset=["value"])
      .assign(value_z=lambda d: (d["value"] - d["value"].mean()) / d["value"].std())
)
print("clean", df_clean.shape)

## Stage 3 — Exploratory statistics

In [ ]:
summary = df_clean.groupby("group")["value"].describe()
summary

## Stage 4 — Load into SQLite  (connection persists across iterations)

In [ ]:
# `conn` is a non-serializable runtime object that survives across agent iterations
conn = sqlite3.connect(":memory:")
df_clean.to_sql("observations", conn, if_exists="replace", index=False)
print("rows in db:", conn.execute("SELECT COUNT(*) FROM observations").fetchone()[0])

## Stage 5 — Parameterized filter  (AGENT EDITS THIS CELL)

In [ ]:
# AGENT EDITS THIS: change `threshold` to iterate
threshold = 5.0
filtered = df_clean[df_clean["value"] > threshold]
print("threshold", threshold, "-> rows", len(filtered))

## Stage 6 — Summary  (depends on the parameter cell)

In [ ]:
result = {
    "n_filtered": int(len(filtered)),
    "mean_value": float(filtered["value"].mean()) if len(filtered) else None,
    "groups": filtered["group"].value_counts().to_dict(),
}
result